In [ ]:
# Run from team-04-modeling/notebooks/. Paths below are relative to the repo root,
# whose data/ folder is git-ignored. NDA: never commit data, outputs or route-level results.
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("../..")

In [ ]:
import glob
import os
import zipfile
import pandas as pd

data_dir = './data'

zip_files = glob.glob(os.path.join(data_dir, '*NY.GDP.PCAP.CD*.zip'))
if not zip_files:
    raise FileNotFoundError("Could not find the NY.GDP.PCAP.CD zip file in your data folder.")

with zipfile.ZipFile(zip_files[0]) as z:
    csv_filename = [f for f in z.namelist() if f.startswith('API_') and f.endswith('.csv')][0]
    with z.open(csv_filename) as f:
        raw_df = pd.read_csv(f, skiprows=4)

year_cols = [col for col in raw_df.columns if col.isdigit() and 2013 <= int(col) <= 2025]
clean_df = raw_df[['Country Code'] + year_cols].copy()

gdp_long = clean_df.melt(
    id_vars=['Country Code'],
    value_vars=year_cols,
    var_name='year',
    value_name='gdp_per_capita'
)

gdp_long = gdp_long.rename(columns={'Country Code': 'iso3_country'})
gdp_long['year'] = gdp_long['year'].astype(int)
gdp_long = gdp_long.dropna(subset=['gdp_per_capita']).sort_values(['iso3_country', 'year'])

output_path = os.path.join(data_dir, 'country_gdp_per_capita.csv')
gdp_long.to_csv(output_path, index=False)
print(f"Saved {len(gdp_long)} rows to {output_path}")
print(gdp_long.head())


In [ ]:
import wbgapi as wb
import pandas as pd

indicators = ['NY.GDP.PCAP.CD', 'SP.POP.TOTL', 'ST.INT.ARVL']

df_wb = wb.data.DataFrame(indicators, time=range(2017, 2026), labels=False).reset_index()

df_long = df_wb.melt(id_vars=['economy', 'series'], var_name='yr', value_name='val')
df_long['year'] = df_long['yr'].str.replace('YR', '').astype(int)

wb_clean = (
    df_long.pivot(index=['economy', 'year'], columns='series', values='val')
    .reset_index()
    .rename(columns={
        'economy': 'iso3_country',
        'NY.GDP.PCAP.CD': 'gdp_per_capita',
        'SP.POP.TOTL': 'country_pop',
        'ST.INT.ARVL': 'intl_tourism_arrivals'
    })
)

wb_clean.to_csv('./data/world_bank_macro.csv', index=False)
print("Saved world_bank_macro.csv:", wb_clean.shape)


In [ ]:
import os
print(os.getcwd())

In [ ]:
import pandas as pd

fred_fx_series = {
    'KOR': 'EXKOUS',  # South Korea (Won per USD)
    'JPN': 'EXJPUS',  # Japan (Yen per USD)
    'GBR': 'EXUSUK',  # UK (USD per GBP)
    'CAN': 'EXCAUS',  # Canada (CAD per USD)
    'MEX': 'EXMXUS',  # Mexico (Pesos per USD)
    'CHN': 'EXCHUS',  # China (Yuan per USD)
    'TWN': 'EXTAUS',  # Taiwan (TWD per USD)
    'IND': 'EXINUS',  # India (Rupees per USD)
    'BRA': 'EXBZUS',  # Brazil (Reais per USD)
    # Eurozone countries all use EXUSEU (USD per EUR)
    'DEU': 'EXUSEU', 'FRA': 'EXUSEU', 'ITA': 'EXUSEU', 'ESP': 'EXUSEU', 'NLD': 'EXUSEU'
}

fx_tables = []
for iso3, series_id in fred_fx_series.items():
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    df = pd.read_csv(url)
    df.columns = ['date', 'fx_rate']
    df['fx_rate'] = pd.to_numeric(df['fx_rate'], errors='coerce')
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['iso3_country'] = iso3
    fx_tables.append(df[['iso3_country', 'year', 'month', 'fx_rate']].dropna())

pd.concat(fx_tables, ignore_index=True).to_csv(r'./data/exchange_rates.csv', index=False)
print("Saved exchange_rates.csv!")


In [ ]:
import calendar
import holidays
import pandas as pd
from dateutil.easter import easter

us_holidays = holidays.US(years=range(2017, 2027))
rows = []

for yr in range(2017, 2029):
    easter_mo = easter(yr).month
    for mo in range(1, 13):
        _, num_days = calendar.monthrange(yr, mo)
        dates = pd.date_range(f"{yr}-{mo:02d}-01", periods=num_days)
        rows.append({
            'flight_year': yr,
            'flight_month': mo,
            'weekend_peak_days': sum(d.weekday() in (4, 6) for d in dates),
            'us_holiday_count': sum(d.date() in us_holidays for d in dates),
            'is_easter_month': int(easter_mo == mo),
            'is_summer_break': int(mo in (6, 7, 8)),
            'is_winter_holiday': int(mo in (11, 12)),
            'is_covid' : int(yr in (2020, 2021))
        })

pd.DataFrame(rows).to_csv(r'./data/calendar_features.csv', index=False)
print("Saved calendar_features.csv!")


In [ ]:
from pathlib import Path
import pandas as pd

folder_path = Path("./data/")

for xlsx_file in folder_path.glob("*.xlsx"):
    # Skip temporary Excel lock files (which start with ~$)
    if xlsx_file.name.startswith("~$"):
        continue

    df = pd.read_excel(xlsx_file)

    csv_file = xlsx_file.with_suffix(".csv")

    df.to_csv(csv_file, index=False, encoding="utf-8")
    print(f"Converted: {xlsx_file.name} -> {csv_file.name}")
